# Classical Fake-News Classification in Google Colab

This notebook trains **only Logistic Regression and calibrated Linear SVM** using the repository's prepared datasets and reproducible TF-IDF pipeline.

The notebook treats model probabilities as content-classification signals—not proof that a story is true or false. Models, metrics, and other generated files are stored in Google Drive so they survive Colab runtime resets.

## Before running

1. Place the existing project `data` folder in `MyDrive/hybrid-fake-news-detection/data`. It must contain `raw`, `interim`, and `processed`, including `processed/manifest.json`.
2. Run the notebook from top to bottom. A CPU runtime is sufficient; GPU is not required.

In [ ]:
from pathlib import Path

REPO_OWNER = "hardiktyagi032"
REPO_NAME = "hybrid-fake-news-detection"
REPO_DIR = Path("/content") / REPO_NAME
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/hybrid-fake-news-detection")
DRIVE_DATA_DIR = DRIVE_PROJECT_DIR / "data"
DRIVE_MODEL_DIR = DRIVE_PROJECT_DIR / "models"
APPROVED_MODELS = ("logistic_regression", "linear_svm")
TRAINING_DATASET = "isot"
EXTERNAL_DATASETS = ("ifnd", "indian_fake_news", "bharatgather")
RANDOM_SEED = 20261017
MAX_FEATURES = 75000

print(f"Repository workspace: {REPO_DIR}")
print(f"Persistent data: {DRIVE_DATA_DIR}")
print(f"Persistent models: {DRIVE_MODEL_DIR}")

## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

required_data_paths = [
    DRIVE_DATA_DIR / "raw",
    DRIVE_DATA_DIR / "interim",
    DRIVE_DATA_DIR / "processed",
    DRIVE_DATA_DIR / "processed" / "manifest.json",
]
missing = [str(path) for path in required_data_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Project data is missing from Google Drive:\n" + "\n".join(missing)
    )
print("Google Drive is mounted and the project data layout is present.")

## 2. Clone the public GitHub repository

In [ ]:
import shutil
import subprocess
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

clone_url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
clone = subprocess.run(
    ["git", "clone", "--depth", "1", clone_url, str(REPO_DIR)],
    capture_output=True,
    text=True,
)
if clone.returncode != 0:
    raise RuntimeError(f"Repository clone failed:\n{clone.stderr}")
print("Public repository cloned successfully.")

## 3. Connect persistent data and model storage

The fresh clone contains only placeholder data directories. This cell links its `data` and `models` paths to the corresponding Google Drive folders.

In [ ]:
import os

def replace_with_directory_link(target: Path, source: Path) -> None:
    if target.is_symlink() or target.is_file():
        target.unlink()
    elif target.is_dir():
        shutil.rmtree(target)
    os.symlink(source, target, target_is_directory=True)

replace_with_directory_link(REPO_DIR / "data", DRIVE_DATA_DIR)
replace_with_directory_link(REPO_DIR / "models", DRIVE_MODEL_DIR)
print("Repository data and model paths now use persistent Drive storage.")

## 4. Install the classical ML environment

In [ ]:
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements-dev.txt")],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements-ml.txt")],
    check=True,
)
print("Dependencies installed.")

## 5. Verify code and prepared data

Training stops if the tests fail, the prepared outputs changed, or the original raw archive hashes no longer match the Phase 4 manifest.

In [ ]:
import os

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
subprocess.run([sys.executable, "scripts/verify_setup.py"], check=True)
subprocess.run(
    [sys.executable, "scripts/validate_prepared_data.py", "--root", str(REPO_DIR)],
    check=True,
)
print("Code tests and dataset integrity checks passed.")

## 6. Train Logistic Regression and Linear SVM

Both candidates are selected using validation macro-F1, refitted on train plus validation, and evaluated once on the held-out ISOT test split.

In [ ]:
train_command = [
    sys.executable,
    "-m",
    "scripts.train_classical",
    "--root",
    str(REPO_DIR),
    "--dataset",
    TRAINING_DATASET,
    "--models",
    *APPROVED_MODELS,
    "--max-features",
    str(MAX_FEATURES),
    "--seed",
    str(RANDOM_SEED),
]
subprocess.run(train_command, check=True)

## 7. Display the ISOT comparison

In [ ]:
import json
import pandas as pd
from IPython.display import display

benchmark_path = DRIVE_MODEL_DIR / "classical" / TRAINING_DATASET / "benchmark.json"
benchmark = json.loads(benchmark_path.read_text(encoding="utf-8"))
rows = []
for split_name, metric_group in [
    ("validation", benchmark["candidate_validation_metrics"]),
    ("held_out_test", benchmark["held_out_test_metrics"]),
]:
    for model_name, metrics in metric_group.items():
        rows.append({
            "split": split_name,
            "model": model_name,
            "accuracy": metrics["accuracy"],
            "balanced_accuracy": metrics["balanced_accuracy"],
            "macro_f1": metrics["f1_macro"],
            "fake_precision": metrics["precision_fake"],
            "fake_recall": metrics["recall_fake"],
            "roc_auc": metrics["roc_auc"],
            "brier_score": metrics["brier_score"],
        })
display(pd.DataFrame(rows).round(4))
print(f"Preferred in-domain baseline: {benchmark['selected_model']}")
print(f"Full report: {benchmark_path}")

## 8. Run frozen cross-dataset evaluation

This is a transfer test—not additional training. It helps expose dataset-specific shortcuts that a high ISOT score can hide.

In [ ]:
external_command = [
    sys.executable,
    "-m",
    "scripts.evaluate_classical",
    "--root",
    str(REPO_DIR),
    "--training-dataset",
    TRAINING_DATASET,
    "--targets",
    *EXTERNAL_DATASETS,
    "--models",
    *APPROVED_MODELS,
]
subprocess.run(external_command, check=True)

In [ ]:
external_path = DRIVE_MODEL_DIR / "classical" / TRAINING_DATASET / "external_evaluation.json"
external = json.loads(external_path.read_text(encoding="utf-8"))
external_rows = []
for dataset_name, dataset_result in external["targets"].items():
    for model_name, metrics in dataset_result["models"].items():
        external_rows.append({
            "target_dataset": dataset_name,
            "model": model_name,
            "records": metrics["records"],
            "accuracy": metrics["accuracy"],
            "balanced_accuracy": metrics["balanced_accuracy"],
            "macro_f1": metrics["f1_macro"],
            "roc_auc": metrics["roc_auc"],
        })
display(pd.DataFrame(external_rows).round(4))
print("Large drops relative to ISOT indicate weak cross-dataset generalization.")
print(f"Full report: {external_path}")

## 9. Optional inference demonstration

Edit the title and article below. The two outputs are model confidence scores learned from ISOT—not verified factuality judgments.

In [ ]:
import joblib
from backend.ml.classical_model import fake_probabilities

example_title = "Official agency releases new report"
example_article = (
    "The agency published an audited report with named sources and links to the underlying records."
)
example_document = f"{example_title}\n\n{example_article}"
prediction_rows = []
for model_name in APPROVED_MODELS:
    model_path = DRIVE_MODEL_DIR / "classical" / TRAINING_DATASET / model_name / "pipeline.joblib"
    model = joblib.load(model_path)
    fake_probability = float(fake_probabilities(model, [example_document])[0])
    prediction_rows.append({
        "model": model_name,
        "fake_probability": fake_probability,
        "predicted_class_at_0.5": "fake" if fake_probability >= 0.5 else "real",
    })
display(pd.DataFrame(prediction_rows).round(4))
print("Warning: this is content classification, not evidence-based fact verification.")

## Outputs

Persistent outputs are stored under `MyDrive/hybrid-fake-news-detection/models/classical/isot/`:

- `logistic_regression/pipeline.joblib` and `metrics.json`
- `linear_svm/pipeline.joblib` and `metrics.json`
- `benchmark.json`
- `external_evaluation.json`

Do not commit model binaries or datasets to Git.